# Member 3 — evaluation & benchmarks on Colab GPU
Runtime → Change runtime type → **GPU**. Run top to bottom. Every result lands in `results/metrics/`
(JSON/CSV) and is logged in `experiments/experiment_log.csv`; copy those into the report, never retype numbers.

In [ ]:
# 1. Setup — set these two paths
REPO_URL = "https://github.com/akashsb2005/AFML-Project.git"   # shared team repo
DRIVE_DIR = "/content/drive/MyDrive/afml-slm"                          # shared folder with data/ and checkpoints/

from google.colab import drive
drive.mount('/content/drive')
!git clone -q $REPO_URL /content/AFML-Project || (cd /content/AFML-Project && git pull -q)
%cd /content/AFML-Project
!pip -q install -r requirements.txt
# data/processed and artifacts/tokenizer come from Member 2 (run src/preprocess.py + src/tokenizer.py, or copy from Drive)
!mkdir -p data artifacts && ln -sfn $DRIVE_DIR/data/processed data/processed && ln -sfn $DRIVE_DIR/artifacts/tokenizer artifacts/tokenizer && ln -sfn $DRIVE_DIR/checkpoints checkpoints_drive
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Contract tests (shapes, causal mask, loss, gradients, overfit) — on Member 1's `src/model.py`

In [ ]:
!python -m pytest -q tests

## 3. Day 2 — hardware baseline / short-run throughput (baseline config only)

In [ ]:
!python src/benchmark.py --configs configs/baseline.yaml --label hw_baseline --steps 20

## 4. E3 — matched efficiency benchmark + sequence-length sweep (same GPU for both variants)

In [ ]:
!python src/benchmark.py --configs configs/baseline.yaml configs/efficient_attention.yaml \
    --label bench_sweep --batch-size 16 --seq-lens 128 256 512 1024 2048 --override context_length=2048
!python src/make_figures.py bench --bench results/metrics/bench_sweep_benchmark.json --out-prefix results/figures/bench

## 5. E1/E3 — held-out perplexity (validation: stride = context and context/2; test: once per final checkpoint)

In [ ]:
CKPTS = {"pre_std": "checkpoints_drive/pre_std_v1.pt", "pre_lin": "checkpoints_drive/pre_lin_v1.pt"}   # edit
for label, ck in CKPTS.items():
    !python src/evaluate.py perplexity --ckpt $ck --label {label}_val --stories data/processed/tiny_stories_validation.jsonl

In [ ]:
# run ONLY for the final checkpoints, once
for label, ck in CKPTS.items():
    !python src/evaluate.py perplexity --ckpt $ck --label {label}_test --stories data/processed/tiny_stories_test.jsonl

## 6. E2 — fixed prompts, response NLL, generations, blind rubric sheet

In [ ]:
!python src/select_eval_prompts.py --test data/processed/dolly_test.jsonl \
    --exclude data/processed/dolly_train.jsonl data/processed/dolly_validation.jsonl \
    --tokenizer artifacts/tokenizer/tokenizer.json --context-length 256 --n 60

In [ ]:
MODELS = {"pre": "checkpoints_drive/pre_std_v1.pt", "sft": "checkpoints_drive/sft_std_v1.pt"}   # edit
for label, ck in MODELS.items():
    !python src/evaluate.py sft-loss --ckpt $ck --label {label} --data data/processed/dolly_test.jsonl --exclude data/processed/dolly_train.jsonl data/processed/dolly_validation.jsonl
    !python src/evaluate.py generate --ckpt $ck --label {label} --prompts results/metrics/eval_prompts.jsonl
    !python src/evaluate.py generate --ckpt $ck --label {label}_sampled --prompts results/metrics/eval_prompts.jsonl --temperature 0.8 --top-k 50
!python src/evaluate.py rubric-sheet --outputs pre=results/metrics/pre_generations.jsonl sft=results/metrics/sft_generations.jsonl

Download `results/metrics/rubric_sheet.csv`, score it **without opening `rubric_key.json`**
(rules in `report/instruction_rubric.md`), upload it back, then:

In [ ]:
!python src/evaluate.py rubric-score --sheet results/metrics/rubric_sheet.csv --key results/metrics/rubric_key.json

## 7. Figures from training logs

In [ ]:
!python src/make_figures.py loss --logs results/metrics/pre_std_v1_train_log.csv results/metrics/pre_lin_v1_train_log.csv \
    --labels "Standard attention" "Linear attention" --out results/figures/pretrain_loss.png

## 8. Log every run (only measured values)

In [ ]:
!python src/log_experiment.py --run-id eval_pre_std_val --member Samvith --stage eval --attention standard \
    --checkpoint checkpoints_drive/pre_std_v1.pt --results-file results/metrics/pre_std_val_perplexity.json